[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eygpcr/biyofizik2026-martini/blob/main/notebooks/06_bentopy.ipynb)

# Oturum 6 — `bentopy` ile Kalabalık Hücresel Sistemler

**Biyofizik 2026 Kursu · 25 Ağustos 2026 · Dr. Öğr. Üyesi Ekrem Yaşar**

Bu not defterinde Martini resmî öğretim materyali uygulanmaktadır:
[Bentopy Tutorial — cgmartini.nl](https://cgmartini.nl/docs/tutorials/Martini3/Bentopy/)

Oturum 4'te tek bir reseptör içeren bir membran sistemi kurulmuştu. Burada
ölçek büyütülmekte ve **çok sayıda protein içeren, bölmeli** hücresel sistemler
inşa edilmektedir.

| Oturum | Sistem | Karakteristik uzunluk |
|---|---|---|
| 2 | Çözelti içinde tek lizozim molekülü | yaklaşık 5 nm |
| 3 | Membranda tek reseptör, atomistik | yaklaşık 10 nm |
| 4 | Membranda tek reseptör, kaba-taneli | yaklaşık 12 nm |
| **6** | **Kalabalık, çok bölmeli sistem** | **40 nm** |

Kursun başlığındaki *"büyük ve kalabalık hücresel sistemler"* ifadesi bu
oturumun konusudur. Kullanılan model proteinlerden biri lizozimdir; Oturum
2'de atomistik olarak hazırlanan proteinin kaba-taneli karşılığı burada
yüzlerce kopya hâlinde kullanılmaktadır.

## İçindekiler

| Bölüm | Konu |
|---|---|
| 1 | Google Drive bağlanması ve çalışma klasörü |
| 2 | Yazılım kurulumu |
| 3 | Görselleştirme araçları |
| 4 | Öğretim dosyalarının indirilmesi ve girdi yapıları |
| 5 | `.bent` yapılandırma biçimi |
| 6 | Uygulama 1 — kutu içinde homojen paketleme |
| 7 | Uygulama 2 — membran çevresinde konuma bağlı paketleme |
| 8 | Uygulama 3 — çok bölmeli sistem |
| 9 | Ölçek karşılaştırması |
| 10 | Simülasyon paketinin hazırlanması |
| 11 | Drive klasörünün özeti |


---
## 1. Google Drive bağlanması

**Aşağıdaki hücre ne yapıyor?** Google Drive'ı bağlayıp klasör yapısını
oluşturmaktadır. Çalıştırıldığında hesap erişim izni istenecektir.

Bu oturumdaki üç uygulama **aynı dosya adlarını** kullanmaktadır
(`placements.json`, `system.gro`, `topol.top`, `solvated_system.gro`). Her
uygulama bir öncekinin çıktısının üzerine yazdığından, her biri kendi alt
klasörüne kaydedilmektedir.

```
Drive'ım/
└── Biyofizik2026_Martini/
    ├── martini_input/              Oturum 4
    └── bentopy/                    bu oturum
        ├── uygulama1_kutu/
        ├── uygulama2_membran/
        ├── uygulama3_bolmeler/
        ├── gorseller/
        └── simulasyon/             GROMACS ile çalıştırılabilir paket
```

> **Dosya boyutu uyarısı.** Solvatlanmış 40 nm'lik sistemler 100 MB'ı
> aşabilmektedir. Üçünü birden Drive'a yazmak hem yavaş hem gereksizdir. Bu
> nedenle büyük koordinat dosyalarından yalnızca **Uygulama 2**'ninki
> kaydedilmekte, diğerleri için yapılandırma dosyaları, yerleşim planları ve
> topolojiler saklanmaktadır. Hepsini kaydetmek isterseniz aşağıdaki
> `BUYUK_DOSYA_KAYDET` değişkenini `True` yapınız.


In [ ]:
from google.colab import drive
import os, shutil, glob

drive.mount('/content/drive')

DRIVE_KOK = '/content/drive/MyDrive/Biyofizik2026_Martini'
OTURUM    = os.path.join(DRIVE_KOK, 'bentopy')
D_UYG1    = os.path.join(OTURUM, 'uygulama1_kutu')
D_UYG2    = os.path.join(OTURUM, 'uygulama2_membran')
D_UYG3    = os.path.join(OTURUM, 'uygulama3_bolmeler')
D_GORSEL  = os.path.join(OTURUM, 'gorseller')
D_SIM     = os.path.join(OTURUM, 'simulasyon')

for d in (DRIVE_KOK, OTURUM, D_UYG1, D_UYG2, D_UYG3, D_GORSEL, D_SIM):
    os.makedirs(d, exist_ok=True)

# Buyuk (>50 MB) koordinat dosyalari da kaydedilsin mi?
BUYUK_DOSYA_KAYDET = False

print('Drive klasoru:', OTURUM)
for d in (D_UYG1, D_UYG2, D_UYG3, D_GORSEL, D_SIM):
    print('  -', os.path.basename(d))
print()
print('Buyuk dosyalar kaydedilecek mi?', BUYUK_DOSYA_KAYDET)


**Aşağıdaki hücre ne yapıyor?** Dosyaları Drive'a kopyalayan yardımcı
fonksiyonu tanımlamaktadır. Oturum 4 ile aynı fonksiyondur; ek olarak boyut
sınırı denetimi yapmaktadır.


In [ ]:
BOYUT_SINIRI = 50 * 1024**2   # 50 MB

def kaydet(desenler, hedef, sessiz=False, buyuk_dahil=None):
    """Verilen dosya desenlerini Drive'daki hedef klasore kopyalar."""
    if buyuk_dahil is None:
        buyuk_dahil = BUYUK_DOSYA_KAYDET
    kopyalanan, atlanan = [], []
    for desen in desenler:
        for dosya in glob.glob(desen):
            if not os.path.isfile(dosya):
                continue
            if os.path.getsize(dosya) > BOYUT_SINIRI and not buyuk_dahil:
                atlanan.append((os.path.basename(dosya),
                                os.path.getsize(dosya) / 1e6))
                continue
            shutil.copy(dosya, hedef)
            kopyalanan.append(os.path.basename(dosya))
    if not sessiz:
        hedef_ad = os.path.basename(hedef)
        if kopyalanan:
            print(f"Drive'a kaydedildi ({hedef_ad}/): "
                  + ', '.join(sorted(kopyalanan)))
        if atlanan:
            print('Boyut sinirini astigi icin atlandi '
                  '(BUYUK_DOSYA_KAYDET=True ile kaydedilir):')
            for a, b in atlanan:
                print(f'  - {a} ({b:.0f} MB)')
    return kopyalanan

print('kaydet() hazir.')


---
## 2. Yazılım kurulumu

**Aşağıdaki hücre ne yapıyor?** `bentopy` ve GROMACS'i kurmaktadır.

`bentopy` Rust ile yazılmış olup PyPI'da yalnızca Linux x86_64 için önceden
derlenmiş paket sunmaktadır. Colab bu platformda çalıştığından kurulum hızlı
tamamlanacaktır. Kendi bilgisayarınızda (özellikle macOS'ta) kurulum, kaynaktan
derleme gerektirebilir; bu durumda [rustup](https://rustup.rs/) ile Rust
derleyicisi kurulmalıdır.


In [ ]:
%%capture
!pip install -q bentopy py3Dmol
!apt-get -qq update && apt-get -qq install -y gromacs


**Aşağıdaki hücre ne yapıyor?** `bentopy`'nin beş alt komutunun da kurulduğunu
doğrulamaktadır. Eksik komut varsa kurulum hücresi yeniden çalıştırılmalıdır.

| Komut | İşlevi |
|---|---|
| `bentopy-mask` | Var olan bir yapıdan bölme maskeleri üretir |
| `bentopy-pack` | Yapıları bölmelere çakışmasız yerleştirir |
| `bentopy-render` | Yerleşim planından koordinat ve topoloji üretir |
| `bentopy-merge` | Paketlenen yapıları var olan bir sistemle birleştirir |
| `bentopy-solvate` | Kalan boşluğu çözücü ve iyonlarla doldurur |


In [ ]:
import shutil as _sh

komutlar = ['bentopy-mask', 'bentopy-pack', 'bentopy-render',
            'bentopy-merge', 'bentopy-solvate']
eksik = [k for k in komutlar if _sh.which(k) is None]
for k in komutlar:
    print(f'  {k:<18} {"bulundu" if _sh.which(k) else "BULUNAMADI"}')
print()
!gmx --version 2>&1 | grep -i 'GROMACS version'
print()
import py3Dmol; print('py3Dmol hazir')
print()
print('DOGRULAMA BASARILI: tum bentopy komutlari hazir.' if not eksik
      else f'UYARI: eksik komutlar -> {eksik}. Kurulum hucresini tekrarlayiniz.')


---
## 3. Görselleştirme araçları

**Aşağıdaki hücre ne yapıyor?** Oturum 4'te kullanılan görselleştirme
fonksiyonlarının aynısını tanımlamaktadır; böylece iki oturumun çizimleri
karşılaştırılabilir olmaktadır. Hiçbir çıktı üretmez.

| Fonksiyon | İşlevi |
|---|---|
| `yapi_goster()` | Yapıyı **etkileşimli** gösterir |
| `koordinat_oku()` | PDB/GRO dosyasından rezidü adı, atom adı ve koordinat okur |
| `bilesen_maskeleri()` | Protein / lipit / çözücü / iyon ayrımı yapar |
| `kesit_ciz()` | **Statik** kesit çizimi üretip PNG kaydeder |
| `z_profili()` | z ekseni boyunca bileşen dağılımını çizer |
| `parcacik_sayisi()` | GRO dosyasındaki parçacık sayısını verir |

> **Ölçek notu.** Bu oturumdaki sistemler milyonlarca parçacık
> içerebilmektedir. Etkileşimli görünüm yalnızca küçük girdi yapıları için
> kullanılmakta; kurulan büyük sistemler `kesit_ciz` ve `z_profili` ile
> incelenmektedir. Bu fonksiyonlar gerektiğinde otomatik örnekleme yapmaktadır.


In [ ]:
import py3Dmol
import numpy as np
import matplotlib.pyplot as plt

LIPIT  = {'POPC','POPE','POPS','POPG','CHOL','DOPC','DPPC','DOPE'}
COZUCU = {'W','WF','PW'}
IYON   = {'NA','CL','NA+','CL-','ION','K','K+'}
RENK   = {'Protein': '#2E5FA3', 'Lipit': '#E08A2E',
          'Cozucu': '#9BC49B', 'Iyon': '#C0392B'}

# --- Ogretim materyalindeki renk duzeni ---
TUT_LIPIT  = '#8D8D8D'
TUT_COZUCU = '#B3E5FC'
TUT_IYON   = '#C0392B'
TUT_PALET  = ['#43A047', '#1E88E5', '#8E24AA', '#00897B']   # yesil, mavi, mor, turkuaz
TUT_BILINEN = {'lyz': ('#43A047', 'Lizozim'),
               'ubq': ('#1E88E5', 'Ubikitin')}


def protein_turleri(res):
    """Sistemdeki protein rezidu adlarini bolluk sirasiyla dondurur.

    Rezidu adlari .bent dosyasindaki segment adiyla ayni olmak zorunda degildir
    (ornegin 'LYZ:lyz' tanimi .gro icinde 'lyz' olarak gorunur). Bu nedenle
    adlar koda sabit yazilmaz, dosyadan okunur.
    """
    diger = LIPIT | COZUCU | IYON
    adlar, sayilar = np.unique(res, return_counts=True)
    p = [(a, int(n)) for a, n in zip(adlar, sayilar) if a not in diger]
    return sorted(p, key=lambda x: -x[1])


def tut_renk_etiket(rn, sira):
    """Bir protein rezidu adi icin renk ve okunabilir etiket secer."""
    for anahtar, (renk, etiket) in TUT_BILINEN.items():
        if anahtar in rn.lower():
            return renk, etiket
    return TUT_PALET[sira % len(TUT_PALET)], rn


def koordinat_oku(dosya, max_atom=400_000):
    """PDB veya GRO dosyasindan rezidu adi, atom adi ve koordinat (nm) okur."""
    if dosya.endswith('.gro'):
        satirlar = open(dosya).read().splitlines()
        n = int(satirlar[1])
        atomlar = satirlar[2:2+n]
        if n > max_atom:
            atomlar = atomlar[::(n // max_atom + 1)]
        res = np.array([s[5:10].strip()  for s in atomlar])
        ad  = np.array([s[10:15].strip() for s in atomlar])
        xyz = np.array([[float(s[20:28]), float(s[28:36]), float(s[36:44])]
                        for s in atomlar])
    else:
        satirlar = [l for l in open(dosya) if l.startswith(('ATOM  ', 'HETATM'))]
        res = np.array([l[17:20].strip() for l in satirlar])
        ad  = np.array([l[12:16].strip() for l in satirlar])
        xyz = np.array([[float(l[30:38]), float(l[38:46]), float(l[46:54])]
                        for l in satirlar]) / 10.0
    return res, ad, xyz


def bilesen_maskeleri(res):
    diger = list(LIPIT | COZUCU | IYON)
    return {'Protein': ~np.isin(res, diger),
            'Lipit'  : np.isin(res, list(LIPIT)),
            'Cozucu' : np.isin(res, list(COZUCU)),
            'Iyon'   : np.isin(res, list(IYON))}


def yapi_goster(dosya, stil='karton', genislik=820, yukseklik=520,
                cozucu_gizle=True, bilgi=True):
    """Yapiyi not defteri icinde etkilesimli olarak gosterir.

    stil='sistem' secildiginde her protein turu ayri renkle gosterilir.
    """
    bicim = 'gro' if dosya.endswith('.gro') else 'pdb'
    v = py3Dmol.view(width=genislik, height=yukseklik)
    v.addModel(open(dosya).read(), bicim)

    if stil == 'kure':
        v.setStyle({}, {'sphere': {'radius': 1.6}})
    elif stil == 'cg_protein':
        v.setStyle({}, {'sphere': {'radius': 1.8, 'color': '#7F8C8D'}})
        v.setStyle({'atom': 'BB'}, {'sphere': {'radius': 2.4, 'color': '#2E5FA3'}})
    elif stil == 'sistem':
        v.setStyle({}, {})
        if not cozucu_gizle:
            v.addStyle({'resn': list(COZUCU)},
                       {'sphere': {'radius': 0.7, 'color': TUT_COZUCU, 'opacity': 0.30}})
        v.addStyle({'resn': list(LIPIT)},
                   {'sphere': {'radius': 1.3, 'color': TUT_LIPIT, 'opacity': 0.55}})
        v.addStyle({'resn': list(IYON)},
                   {'sphere': {'radius': 1.4, 'color': TUT_IYON, 'opacity': 0.7}})
        # Her protein turu ayri renkte
        res, _, _ = koordinat_oku(dosya)
        turler = protein_turleri(res)
        if bilgi:
            print('Renk atamasi:')
        for k, (rn, sayi) in enumerate(turler):
            renk, etiket = tut_renk_etiket(rn, k)
            v.addStyle({'resn': rn},
                       {'sphere': {'radius': 2.4, 'color': renk}})
            if bilgi:
                print(f"  {etiket:<12} ('{rn}')  {renk}   {sayi:,} merkez")
        if bilgi and any(np.isin(res, list(LIPIT))):
            print(f"  {'Membran':<12} ('POPC')     {TUT_LIPIT}   lipit fosfat merkezleri")
    else:
        v.setStyle({}, {'cartoon': {'color': 'spectrum'}})

    v.zoomTo(); v.setBackgroundColor('white')
    return v.show()



def kesit_ciz(dosya, png, baslik, dilim=None, eksen=('x', 'z')):
    """Yapinin izdusum gorunumunu cizer ve PNG olarak kaydeder."""
    res, ad, xyz = koordinat_oku(dosya)
    i = {'x': 0, 'y': 1, 'z': 2}[eksen[0]]
    j = {'x': 0, 'y': 1, 'z': 2}[eksen[1]]
    k = 3 - i - j
    sec = np.ones(len(res), dtype=bool)
    if dilim is not None:
        orta = (xyz[:, k].min() + xyz[:, k].max()) / 2
        sec = np.abs(xyz[:, k] - orta) < dilim / 2
    maskeler = bilesen_maskeleri(res)
    plt.figure(figsize=(7.5, 7.5))
    for ad_g, boyut, saydam in [('Cozucu', 1.0, 0.15), ('Lipit', 2.5, 0.55),
                                ('Iyon', 5.0, 0.75), ('Protein', 3.5, 0.9)]:
        m = maskeler[ad_g] & sec
        if m.sum() == 0:
            continue
        plt.scatter(xyz[m, i], xyz[m, j], s=boyut, c=RENK[ad_g], alpha=saydam,
                    linewidths=0,
                    label=f'{ad_g} ({m.sum():,} / {maskeler[ad_g].sum():,})')
    plt.xlabel(f'{eksen[0]} (nm)'); plt.ylabel(f'{eksen[1]} (nm)')
    plt.title(baslik); plt.gca().set_aspect('equal')
    plt.legend(loc='upper right', framealpha=.9, markerscale=4,
               title='dilimde / toplam', title_fontsize=8, fontsize=8)
    plt.grid(alpha=.2); plt.tight_layout()
    plt.savefig(png, dpi=150); plt.show()
    print('Kaydedildi:', png)


def z_profili(dosya, png, baslik):
    res, ad, xyz = koordinat_oku(dosya)
    z = xyz[:, 2]
    maskeler = bilesen_maskeleri(res)
    kenar = np.linspace(z.min(), z.max(), 120)
    plt.figure(figsize=(9, 4.5))
    for ad_g, m in maskeler.items():
        if m.sum() == 0:
            continue
        plt.hist(z[m], bins=kenar, histtype='step', lw=1.7,
                 color=RENK[ad_g], label=f'{ad_g} (n={m.sum():,})')
    plt.xlabel('z (nm)'); plt.ylabel('Parcacik sayisi')
    plt.title(baslik); plt.legend(); plt.grid(alpha=.3)
    plt.tight_layout(); plt.savefig(png, dpi=150); plt.show()
    print('Kaydedildi:', png)


def parcacik_sayisi(gro):
    return int(open(gro).read().splitlines()[1])


def boyut_str(bayt):
    """Dosya boyutunu uygun birimde bicimlendirir."""
    if bayt < 1024:
        return f'{bayt} B'
    if bayt < 1024**2:
        return f'{bayt/1024:.1f} KB'
    return f'{bayt/1024**2:.1f} MB'


def boyut_yaz(*dosyalar):
    for d in dosyalar:
        if os.path.exists(d):
            print(f'  {os.path.basename(d):<30} {boyut_str(os.path.getsize(d)):>10}')
        else:
            print(f'  {os.path.basename(d):<30} {"BULUNAMADI":>10}')

print('Gorsellestirme fonksiyonlari hazir.')


**Aşağıdaki hücre ne yapıyor?** Büyük sistemler için iki ek görselleştirme
aracı tanımlamaktadır. Oturum 4'teki sistem 17 bin parçacıktı ve doğrudan
etkileşimli gösterilebiliyordu; bu oturumdaki sistemler milyonlarca parçacık
içerdiğinden aynı yöntem tarayıcıyı kilitler.

| Fonksiyon | İşlevi |
|---|---|
| `seyreltilmis_gro()` | Sistemden etkileşimli gösterime uygun küçük bir örnek üretir: proteinlerden yalnızca omurga (BB), lipitlerden yalnızca fosfat (PO4) merkezleri alınır ve bir kısmı örneklenir |
| `molekul_merkezleri()` | Her molekülün ağırlık merkezini hesaplar |
| `uc_boyut_goster()` | Molekül merkezlerini üç boyutlu perspektifle çizer — kalabalık kutunun bütününü tek bakışta gösterir |

Böylece her uygulamada yapı üç şekilde incelenmektedir: etkileşimli seyreltilmiş
görünüm, üç boyutlu perspektif ve iki boyutlu kesit.


In [ ]:
def seyreltilmis_gro(girdi, cikti, protein_orani=0.15, lipit_orani=0.06,
                     cozucu_dahil=False):
    """Buyuk bir sistemden py3Dmol ile gosterilebilecek kucuk bir ornek uretir.

    Proteinlerden yalnizca omurga (BB) merkezleri, lipitlerden yalnizca fosfat
    (PO4) merkezleri alinir ve bunlarin bir kismi ornekleneir.
    """
    satirlar = open(girdi).read().splitlines()
    n = int(satirlar[1])
    atomlar = satirlar[2:2+n]
    kutu = satirlar[2+n] if len(satirlar) > 2+n else '  40.0  40.0  40.0'

    secilen = []
    p_sayac = l_sayac = 0
    # Oran 0 ise o bilesen tamamen atlanir (orn. Uygulama 1'de lipit yoktur)
    p_adim = max(int(round(1 / protein_orani)), 1) if protein_orani > 0 else 0
    l_adim = max(int(round(1 / lipit_orani)), 1) if lipit_orani > 0 else 0
    for s in atomlar:
        rn, an = s[5:10].strip(), s[10:15].strip()
        if rn in COZUCU:
            if cozucu_dahil and (p_sayac + l_sayac) % 97 == 0:
                secilen.append(s)
            continue
        if rn in IYON:
            continue
        if rn in LIPIT:
            if l_adim and an == 'PO4':
                if l_sayac % l_adim == 0:
                    secilen.append(s)
                l_sayac += 1
            continue
        if p_adim and an == 'BB':
            if p_sayac % p_adim == 0:
                secilen.append(s)
            p_sayac += 1

    with open(cikti, 'w') as f:
        f.write('Gorsellestirme icin seyreltilmis sistem\n')
        f.write(f'{len(secilen)}\n')
        f.write('\n'.join(secilen) + '\n')
        f.write(kutu + '\n')
    print(f'{girdi} -> {cikti}: {n:,} parcacikdan {len(secilen):,} tanesi secildi')
    return cikti


def molekul_merkezleri(gro, resnames=None, max_molekul=4000):
    """Her molekulun agirlik merkezini dondurur: {resname: (N,3) dizi}."""
    satirlar = open(gro).read().splitlines()
    n = int(satirlar[1])
    gruplar, aktif = {}, None
    son_resid = None
    for s in satirlar[2:2+n]:
        rn = s[5:10].strip()
        if rn in COZUCU or rn in IYON:
            continue
        if resnames and rn not in resnames:
            continue
        resid = int(s[0:5])
        xyz = [float(s[20:28]), float(s[28:36]), float(s[36:44])]
        anahtar = (rn, resid)
        if anahtar != son_resid:
            aktif = gruplar.setdefault(rn, [])
            aktif.append([xyz, 1])
            son_resid = anahtar
        else:
            aktif[-1][0] = [a + b for a, b in zip(aktif[-1][0], xyz)]
            aktif[-1][1] += 1
    sonuc = {}
    for rn, lst in gruplar.items():
        m = np.array([[x / k for x in toplam] for toplam, k in lst])
        if len(m) > max_molekul:
            m = m[:: len(m) // max_molekul + 1]
        sonuc[rn] = m
    return sonuc


def kutu_oku(gro):
    """GRO dosyasinin son satirindaki kutu vektorlerini dondurur (nm)."""
    try:
        v = [float(x) for x in open(gro).read().splitlines()[-1].split()[:3]]
        return v if len(v) == 3 else None
    except Exception:
        return None


def uc_boyut_goster(gro, png, baslik, lipit_goster=True,
                    yukselti=16, azimut=-62, kutu_ciz=True):
    """Molekul merkezlerini uc boyutlu perspektifle cizer."""
    from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

    merkezler = molekul_merkezleri(gro)
    proteinler = sorted(((rn, m) for rn, m in merkezler.items() if rn not in LIPIT),
                        key=lambda x: -len(x[1]))
    lipitler = [(rn, m) for rn, m in merkezler.items() if rn in LIPIT]

    fig = plt.figure(figsize=(9.5, 8.5))
    eks = fig.add_subplot(111, projection='3d')

    # --- Panelleri sadelestir ---
    for eksen in (eks.xaxis, eks.yaxis, eks.zaxis):
        eksen.pane.set_facecolor('#FCFCFD')
        eksen.pane.set_edgecolor('#E4E4E8')
        eksen.pane.set_alpha(1.0)
        eksen._axinfo['grid'].update(color='#E8E8EC', linewidth=0.6)
        eksen.line.set_color('#BBBBC2')
    eks.tick_params(colors='#555555', labelsize=9)

    # --- Simulasyon kutusunun tel cerceve gosterimi ---
    kutu = kutu_oku(gro)
    if kutu_ciz and kutu:
        X, Y, Z = kutu
        kenarlar = [((0,0,0),(X,0,0)), ((0,0,0),(0,Y,0)), ((0,0,0),(0,0,Z)),
                    ((X,Y,Z),(0,Y,Z)), ((X,Y,Z),(X,0,Z)), ((X,Y,Z),(X,Y,0)),
                    ((X,0,0),(X,Y,0)), ((X,0,0),(X,0,Z)),
                    ((0,Y,0),(X,Y,0)), ((0,Y,0),(0,Y,Z)),
                    ((0,0,Z),(X,0,Z)), ((0,0,Z),(0,Y,Z))]
        for (a, b) in kenarlar:
            eks.plot(*zip(a, b), color='#9AA0A6', lw=0.7, alpha=.55, zorder=0)

    # --- Lipitler: arka planda ince bir tabaka ---
    if lipit_goster:
        for rn, m in lipitler:
            eks.scatter(m[:, 0], m[:, 1], m[:, 2], s=2.6, c=TUT_LIPIT,
                        alpha=.28, linewidths=0, depthshade=False,
                        label=f'Membran ({len(m):,})', zorder=1)

    # --- Proteinler: bolluk sirasina gore ogretim materyali renkleriyle ---
    for k, (rn, m) in enumerate(proteinler):
        renk, etiket = tut_renk_etiket(rn, k)
        # depthshade=False: uzaktaki molekullerin rengi solmasin, iki protein
        # turu her derinlikte ayirt edilebilsin
        eks.scatter(m[:, 0], m[:, 1], m[:, 2], s=44, c=renk, alpha=.92,
                    edgecolors='white', linewidths=.5, depthshade=False,
                    label=f'{etiket} ({len(m):,})', zorder=3)

    eks.set_xlabel('x (nm)', labelpad=8, color='#333333')
    eks.set_ylabel('y (nm)', labelpad=8, color='#333333')
    eks.set_zlabel('z (nm)', labelpad=8, color='#333333')
    eks.set_title(baslik, fontsize=12, fontweight='bold', pad=14)
    eks.view_init(elev=yukselti, azim=azimut)
    eks.set_box_aspect((1, 1, 1))
    if kutu:
        eks.set_xlim(0, kutu[0]); eks.set_ylim(0, kutu[1]); eks.set_zlim(0, kutu[2])

    gosterge = eks.legend(loc='upper left', fontsize=9, markerscale=1.3,
                          framealpha=.92, borderpad=.7, labelspacing=.6)
    gosterge.get_frame().set_edgecolor('#DDDDDD')

    plt.tight_layout()
    plt.savefig(png, dpi=150, facecolor='white'); plt.show()
    print('Kaydedildi:', png)


print('Buyuk sistem gorsellestirme fonksiyonlari hazir.')


**Aşağıdaki hücre ne yapıyor?** Kurduğumuz sistemi öğretim materyalindeki
referans görüntüyle **yan yana** karşılaştıran fonksiyonları tanımlamaktadır.

cgmartini.nl sayfasındaki görüntüler VMD ile üretilmiş yüksek kaliteli
render'lardır. Colab ortamında VMD bulunmadığından ve sistemler milyonlarca
parçacık içerdiğinden aynı kalitede render üretilememektedir. Bunun yerine iki
boyutlu kesit çizimi kullanılmakta, referans görüntü yanına konarak **neyin
görülmesi gerektiği** açıkça gösterilmektedir.

Karşılaştırmanın anlamlı olması için kesit çizimi öğretim materyalindeki renk
düzenine göre yapılmaktadır: yeşil lizozim, mavi ubikitin, gri membran, açık
mavi su.

> **Rezidü adları hakkında.** `.bent` dosyasındaki segment adı ile koordinat
> dosyasındaki rezidü adı aynı olmak zorunda değildir; `LYZ:lyz` tanımı `.gro`
> içinde `lyz` olarak görünmektedir. Bu nedenle protein adları koda sabit
> yazılmamakta, dosyadan okunup bolluk sırasına göre renklendirilmektedir.
> Fonksiyon bulduğu adları ayrıca ekrana yazmaktadır.


In [ ]:
import os
import urllib.request
import matplotlib.image as mpimg

TUT_TABAN = 'https://cgmartini.nl/docs/tutorials/Martini3/Bentopy'


def kesit_tutorial_renkli(eks, gro, dilim=6.0, su_goster=True, bilgi=True):
    """Kesiti ogretim materyalindeki renk duzeniyle verilen eksene cizer."""
    res, ad, xyz = koordinat_oku(gro)
    orta = (xyz[:, 1].min() + xyz[:, 1].max()) / 2
    sec = np.abs(xyz[:, 1] - orta) < dilim / 2

    if su_goster:
        m = np.isin(res, list(COZUCU)) & sec
        if m.sum():
            eks.scatter(xyz[m, 0], xyz[m, 2], s=1.0, c=TUT_COZUCU, alpha=0.10,
                        linewidths=0, label=f'Su ({m.sum():,})')

    m = np.isin(res, list(LIPIT)) & sec
    if m.sum():
        eks.scatter(xyz[m, 0], xyz[m, 2], s=2.5, c=TUT_LIPIT, alpha=0.60,
                    linewidths=0, label=f'Membran ({m.sum():,})')

    turler = protein_turleri(res)
    if bilgi:
        print('Dosyada bulunan protein rezidu adlari:',
              ', '.join(f'{a} ({n:,})' for a, n in turler) or 'yok')
    for k, (rn, toplam) in enumerate(turler):
        m = (res == rn) & sec
        if m.sum() == 0:
            continue
        renk, etiket = tut_renk_etiket(rn, k)
        eks.scatter(xyz[m, 0], xyz[m, 2], s=4.0, c=renk, alpha=0.90,
                    linewidths=0, label=f'{etiket} ({m.sum():,})')

    eks.set_xlabel('x (nm)'); eks.set_ylabel('z (nm)')
    eks.set_aspect('equal'); eks.grid(alpha=.15)
    eks.legend(loc='upper right', fontsize=8, markerscale=4, framealpha=.9)


def referans_karsilastir(gro, referans_dosya, png, baslik, dilim=6.0):
    """Kurdugumuz sistemi ogretim materyalindeki referans goruntuyle karsilastirir."""
    if not os.path.exists(referans_dosya):
        try:
            urllib.request.urlretrieve(f'{TUT_TABAN}/{referans_dosya}', referans_dosya)
        except Exception as e:
            print('Referans goruntu indirilemedi:', e)
            referans_dosya = None

    fig, eks = plt.subplots(1, 2, figsize=(15, 8))
    kesit_tutorial_renkli(eks[0], gro, dilim=dilim)
    eks[0].set_title('Bu not defterinde uretilen\n(iki boyutlu kesit)',
                     fontsize=11, fontweight='bold')
    if referans_dosya and os.path.exists(referans_dosya):
        eks[1].imshow(mpimg.imread(referans_dosya))
        eks[1].set_title('cgmartini.nl ogretim materyali\n(VMD render)',
                         fontsize=11, fontweight='bold')
    else:
        eks[1].text(.5, .5, 'Referans goruntu yok', ha='center', va='center')
    eks[1].axis('off')
    fig.suptitle(baslik, fontsize=13, y=0.99)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.savefig(png, dpi=140, facecolor='white'); plt.show()
    print('Kaydedildi:', png)


print('Referans karsilastirma fonksiyonlari hazir.')


---
## 4. Öğretim dosyalarının indirilmesi

**Aşağıdaki hücre ne yapıyor?** cgmartini.nl tarafından sağlanan öğretim
arşivini indirip açmaktadır (yaklaşık 3.6 MB). Arşiv, uygulamalarda
kullanılacak yapıları, topolojileri ve simülasyon parametre dosyalarını
içermektedir.

Tüm `bentopy` komutları `tutorial_files/` dizini içinden çalıştırılmalıdır;
`.bent` dosyalarındaki yollar buna göre yazılmıştır.


In [ ]:
os.chdir('/content')
!wget -q https://cgmartini-library.s3.ca-central-1.amazonaws.com/0_Tutorials/m3_tutorials/Bentopy/tutorial_files.tar.gz
!tar -xzf tutorial_files.tar.gz

CALISMA = '/content/tutorial_files'
os.chdir(CALISMA)
print('Calisma dizini:', os.getcwd())
print()
!ls structures/ topology/ mdp_files/


**Aşağıdaki hücre ne yapıyor?** Girdi yapılarının boyutlarını listelemektedir.

| Yapı | Tanım |
|---|---|
| `lysozyme.pdb` | Lizozim — Oturum 2'deki proteinin kaba-taneli karşılığı |
| `ubiquitin.pdb` | Ubikitin — membran yüzeyine yerleştirilecek ikinci protein |
| `membrane.gro` | Önceden kurulmuş POPC çift tabaka |
| `double_membrane.gro` | İki bölmeyi ayıran çift membran |


In [ ]:
for f in ('lysozyme.pdb', 'ubiquitin.pdb'):
    n = sum(1 for l in open(f'structures/{f}') if l.startswith(('ATOM  ','HETATM')))
    print(f'  {f:<24} {n:>8,} etkilesim merkezi')
for f in ('membrane.gro', 'double_membrane.gro'):
    print(f'  {f:<24} {parcacik_sayisi(f"structures/{f}"):>8,} parcacik')


**Aşağıdaki hücre ne yapıyor?** Lizozimin kaba-taneli yapısını etkileşimli
göstermektedir. Bu tek molekül, Uygulama 1'de 650 kopya hâlinde
kullanılacaktır.


In [ ]:
yapi_goster('structures/lysozyme.pdb', stil='cg_protein')


**Aşağıdaki hücre ne yapıyor?** Ubikitini göstermektedir. Uygulama 2 ve 3'te
membran yüzeyine yakın bölgeye yerleştirilecektir.


In [ ]:
yapi_goster('structures/ubiquitin.pdb', stil='cg_protein')


**Aşağıdaki hücre ne yapıyor?** Hazır POPC çift tabakasının kesitini çizmekte
ve membranın konumunu ölçmektedir. Uygulama 2'de bu membran, paketlenen
proteinlerle birleştirilecektir.


In [ ]:
kesit_ciz('structures/membrane.gro',
          os.path.join(D_GORSEL, '00_hazir_membran.png'),
          'Girdi yapisi - hazir POPC cift tabaka', dilim=4.0, eksen=('x', 'z'))

res, ad, xyz = koordinat_oku('structures/membrane.gro')
po4 = np.isin(res, list(LIPIT)) & (ad == 'PO4')
if po4.sum():
    z = xyz[po4, 2]; orta = z.mean()
    ust, alt = z[z > orta], z[z < orta]
    print()
    print(f'Lipit sayisi     : ust {len(ust)}, alt {len(alt)}')
    print(f'Membran merkezi  : z = {orta:.1f} nm')
    print(f'Membran kalinligi: {ust.mean()-alt.mean():.2f} nm')


---
## 5. `.bent` yapılandırma biçimi

`bentopy` ne isteneceğini `.bent` uzantılı bir metin dosyasından okumaktadır.
Dosya beş bölümden oluşur:

| Bölüm | İçeriği |
|---|---|
| `[ general ]` | Sistem başlığı ve rastgelelik tohumu (`seed`) |
| `[ space ]` | Kutu boyutları ve paketleme ızgara çözünürlüğü |
| `[ includes ]` | Üretilecek topolojiye eklenecek kuvvet alanı dosyaları |
| `[ compartments ]` | Yerleştirmenin yapılacağı hacimlerin tanımı |
| `[ segments ]` | Hangi yapıdan kaç kopyanın hangi bölmeye konacağı |

### Bölme tanımlama dili

`[ compartments ]` bölümü bu aracın en ayırt edici özelliğidir. Hacimler
birbirinden türetilebilmektedir:

| İfade | Anlamı |
|---|---|
| `system is all` | Kutunun tamamı |
| `membrane from "maske.npz"` | Maske dosyasından tanımlanan hacim |
| `solvent combines not membrane` | Membran dışında kalan hacim |
| `yakin around 5 of membrane` | Membran yüzeyinden 5 nm mesafedeki kabuk |
| `X combines A and B` | İki hacmin kesişimi |

Bu yaklaşım, proteinlerin yalnızca sayıca değil **konumsal kurala göre** de
yerleştirilmesini sağlamaktadır: örneğin periferik bir membran proteininin
yalnızca membran yakınında bulunması.

Tam söz dizimi:
[Reference for `.bent` files](https://github.com/marrink-lab/bentopy/wiki/Reference-for-bent)

### Üç aşamalı iş akışı

```
  .bent  ──bentopy-pack──▶  placements.json  ──bentopy-render──▶  .gro + .top
                                                                      │
                              (membran varsa) bentopy-merge ◀─────────┘
                                                                      │
                                          bentopy-solvate ◀───────────┘
```

`bentopy-pack` yalnızca **nereye ne konacağını** hesaplar; koordinat üretmez.
Bu ayrım, aynı yerleşim planının farklı çözünürlüklerde işlenebilmesini
sağlamaktadır.


---
## 6. Uygulama 1 — Kutu içinde homojen paketleme

40 × 40 × 40 nm boyutlarında bir kutuya **650 lizozim** molekülü, sitoplazmik
derişime karşılık gelen yoğunlukta yerleştirilmektedir.

### Kalabalık ortam koşullarının önemi

Moleküler simülasyonların büyük bölümü proteinleri seyreltik çözelti
koşullarında incelemektedir. Hücre içi ortam bu varsayımdan belirgin biçimde
ayrılmaktadır:

- Sitoplazmada toplam makromolekül derişimi yaklaşık **300 g/L** düzeyindedir;
  hacmin %20–30'u makromoleküller tarafından işgal edilmektedir
- Kalabalık koşulları difüzyonu yavaşlatmakta, bağlanma dengelerini
  kaydırmakta, katlanma ve oligomerleşme süreçlerini etkilemektedir

Bu nedenle seyreltik koşullarda yürütülen simülasyonlar hücre içi süreçleri
sistematik biçimde eksik temsil edebilmektedir.

**Aşağıdaki hücre ne yapıyor?** Uygulama 1'in yapılandırma dosyasını
yazmaktadır. `system is all` ifadesi kutunun tamamını tek bir bölme olarak
tanımlamakta; proteinler bu hacme homojen dağıtılmaktadır.


In [ ]:
bent = '''[ general ]
title "Proteins in a box"
seed 0

[ space ]
dimensions 40, 40, 40
resolution 0.5

[ includes ]
"topology/martini_v3.0.0.itp"
"topology/martini_v3.0.0_ions_v1.itp"
"topology/martini_v3.0.0_solvents_v1.itp"
"topology/lysozyme.itp"

[ compartments ]
system is all

[ segments ]
LYZ 650 from "structures/lysozyme.pdb" in system
'''
open('simple_packing.bent', 'w').write(bent)
print(bent)


**Aşağıdaki hücre ne yapıyor?** Yerleşim planını hesaplamaktadır. Bu aşamada
koordinat üretilmez; yalnızca her kopyanın konumu ve yönelimi belirlenir.
İşlem birkaç dakika sürebilir.

Çıktıdaki özet, kaç kopyanın yerleştirilebildiğini göstermektedir. Hedeflenen
sayıya ulaşılamaması kutunun dolduğu anlamına gelir.


In [ ]:
!bentopy-pack simple_packing.bent placements.json


**Aşağıdaki hücre ne yapıyor?** Yerleşim planından koordinat ve topoloji
dosyalarını üretmektedir. `-t` bayrağı topoloji yazılmasını sağlar.


In [ ]:
!bentopy-render placements.json system.gro -t topol.top

print()
print(f'Paketlenmis sistem: {parcacik_sayisi("system.gro"):,} parcacik')
boyut_yaz('system.gro', 'topol.top')


**Aşağıdaki hücre ne yapıyor?** Kalan boşluğu çözücü ve iyonlarla
doldurmaktadır.

| Bayrak | İşlevi |
|---|---|
| `-i` / `-o` | Girdi ve çıktı koordinat dosyaları |
| `-s NA:0.15M -s CL:0.15M` | Çözücünün bir kısmını iyonla değiştirir |
| `--charge neutral` | Sistemin net yükünü sıfırlar |
| `-t` | Topolojiyi günceller |

> **Oturum 4 ile karşılaştırma.** `insane` iyonları `NA+`/`CL-` adlarıyla
> üretiyordu ve Martini 3 ile uyumsuzluk nedeniyle elle düzeltme gerekiyordu.
> `bentopy-solvate` doğrudan `NA`/`CL` adlarını kullanmaktadır; ek düzeltme
> gerekmemektedir.

İşlem birkaç dakika sürebilir ve büyük bir dosya üretir.


In [ ]:
!bentopy-solvate -i system.gro -o solvated_system.gro \
    -s NA:0.15M -s CL:0.15M \
    --charge neutral \
    -t topol.top


**Aşağıdaki hücre ne yapıyor?** Sonucu özetleyip topolojiyi göstermektedir.


In [ ]:
print(f'Solvatlanmis sistem: {parcacik_sayisi("solvated_system.gro"):,} parcacik')
boyut_yaz('solvated_system.gro')
print()
print('--- topol.top ---')
print(open('topol.top').read())

res, ad, xyz = koordinat_oku('solvated_system.gro')
iyon = np.isin(res, ['NA', 'CL'])
print(f'Iyon adlandirmasi: {sorted(set(res[iyon]))} '
      f'-> Martini 3 ile uyumlu' if iyon.sum() else 'Iyon bulunamadi')


**Aşağıdaki hücre ne yapıyor?** Sistemden etkileşimli gösterime uygun
seyreltilmiş bir örnek üretip göstermektedir. Görünümü fare ile döndürerek
kalabalığı inceleyiniz.

Her protein türü **ayrı renkte** gösterilmektedir; renk ataması hücrenin
çıktısında listelenmektedir. Öğretim materyaliyle aynı düzen kullanılmaktadır:
yeşil lizozim, mavi ubikitin, gri membran.

> Bu, sistemin **örneklenmiş** bir gösterimidir; proteinlerden yalnızca omurga,
> lipitlerden yalnızca fosfat merkezleri alınmakta ve bunların bir kısmı
> çizilmektedir. Gerçek sistem çok daha yoğundur; tam sistem iki ve üç boyutlu
> çizimlerle incelenmektedir.


In [ ]:
seyreltilmis_gro('system.gro', 'gorunum_u1.gro',
                 protein_orani=0.1, lipit_orani=0.0)
yapi_goster('gorunum_u1.gro', stil='sistem')


**Aşağıdaki hücre ne yapıyor?** Her molekülün ağırlık merkezini hesaplayıp üç
boyutlu perspektifle çizmektedir. Bu görünüm, moleküllerin kutu içindeki
dağılımını tek bakışta göstermektedir.

Renk düzeni öğretim materyaliyle aynıdır: yeşil lizozim, mavi ubikitin, gri
membran. Simülasyon kutusu tel çerçeve olarak gösterilmektedir.


In [ ]:
uc_boyut_goster('system.gro',
                os.path.join(D_GORSEL, 'uygulama1_3boyut.png'),
                'Uygulama 1 - molekul merkezleri (uc boyutlu)')


**Aşağıdaki hücre ne yapıyor?** Sistemin kesitini çizmektedir.

**Beklenen görünüm:** protein z ekseni boyunca **düzgün dağılmış** olmalıdır.
Uygulama 1'in amacı homojen paketlemedir; herhangi bir bölgede zenginleşme
görülmemelidir.


In [ ]:
kesit_ciz('solvated_system.gro',
          os.path.join(D_GORSEL, 'uygulama1_kesit.png'),
          'Uygulama 1 - kutuda homojen protein paketlemesi (yandan kesit)',
          dilim=6.0, eksen=('x', 'z'))


**Aşağıdaki hücre ne yapıyor?** z ekseni boyunca dağılımı çizmektedir. Protein
eğrisi düz (yatay) olmalıdır — bu, homojen dağılımın niceliksel kanıtıdır.


In [ ]:
z_profili('solvated_system.gro',
          os.path.join(D_GORSEL, 'uygulama1_z_profili.png'),
          'Uygulama 1 - z ekseni boyunca dagilim')

res, ad, xyz = koordinat_oku('solvated_system.gro')
m = bilesen_maskeleri(res)['Protein']
sayim, _ = np.histogram(xyz[m, 2], bins=10)

bagil   = sayim.std() / sayim.mean() * 100
poisson = 100 / np.sqrt(sayim.mean())   # rastgele yerlesimde beklenen dalgalanma

print()
print(f'Dilim basina ortalama parcacik : {sayim.mean():,.0f}')
print(f'Olculen bagil degisim          : %{bagil:.1f}')
print(f'Rastgele yerlesim beklentisi   : %{poisson:.1f}  (Poisson gurultusu)')
print()
if bagil < 2.5 * poisson:
    print('DEGERLENDIRME: dalgalanma rastgele yerlesim beklentisiyle uyumlu;')
    print('dagilim homojendir.')
else:
    print('UYARI: dalgalanma beklenenden buyuk. Kutu sinirlarindaki etki veya')
    print('paketlemenin doymus olmasi soz konusu olabilir.')


**Aşağıdaki hücre ne yapıyor?** Kurduğumuz sistemi öğretim
materyalindeki referans görüntüyle yan yana koymaktadır. Soldaki çizim
bu not defterinde üretilmiştir; sağdaki, cgmartini.nl sayfasındaki VMD
render'ıdır.

Referans görüntüde lizozim molekülleri (yeşil) kutu boyunca **düzgün**
dağılmış durumdadır; herhangi bir bölgede yığılma yoktur. Kendi kesitinizde
de aynı homojenlik görülmelidir.


In [ ]:
referans_karsilastir('solvated_system.gro',
                     'figure_tutorial_1.png',
                     os.path.join(D_GORSEL, 'uygulama1_karsilastirma.png'),
                     'Uygulama 1 - kutuda homojen protein paketlemesi')


**Aşağıdaki hücre ne yapıyor?** Uygulama 1 dosyalarını Drive'a
kaydetmektedir. Solvatlanmış koordinat dosyası boyut sınırını aştığından
varsayılan olarak atlanmaktadır (bkz. 1. bölüm).


In [ ]:
kaydet(['simple_packing.bent', 'placements.json', 'system.gro',
        'topol.top', 'solvated_system.gro'], D_UYG1)


---
## 7. Uygulama 2 — Membran çevresinde konuma bağlı paketleme

Bu uygulamada proteinler yalnızca sayıca değil, **konumsal kurala göre** de
yerleştirilmektedir: lizozim çözücü hacmine dağıtılırken, ubikitin yalnızca
membran yüzeyine yakın bölgede konumlandırılmaktadır. Bu, periferik membran
proteinlerinin fizyolojik dağılımının modellenmesine karşılık gelmektedir.

**Aşağıdaki hücre ne yapıyor?** Membran yapısından bölme maskesi
üretmektedir. İki komut çalıştırılır:

- İlki (`--visualize-labels`) etiketlemenin **görsel olarak denetlenmesi**
  için bir yapı dosyası yazar
- İkincisi (`-l 1:...`) asıl kullanılacak maske dosyasını üretir

Maske, kutunun hangi bölgesinin "membran" sayılacağını tanımlayan üç boyutlu
bir ızgaradır.


In [ ]:
!bentopy-mask structures/membrane.gro --visualize-labels labels.gro
!bentopy-mask structures/membrane.gro -l 1:membrane_mask.npz
print()
boyut_yaz('membrane_mask.npz', 'labels.gro')
print()
print('DOGRULAMA BASARILI: maske uretildi.'
      if os.path.exists('membrane_mask.npz') else
      'UYARI: maske uretilemedi.')


**Aşağıdaki hücre ne yapıyor?** Uygulama 2'nin yapılandırma dosyasını
yazmaktadır. `[ compartments ]` bölümündeki üç tanıma dikkat ediniz:

```
membrane from "membrane_mask.npz"      maskeden tanimlanan hacim
solvent combines not membrane          membran disinda kalan hacim
close-to-membrane around 5 of membrane membran yuzeyinden 5 nm'lik kabuk
```

`[ segments ]` bölümünde her protein farklı bir bölmeye atanmaktadır:
300 lizozim `solvent` içine, 100 ubikitin `close-to-membrane` içine.


In [ ]:
bent = '''[ general ]
title "Proteins around a membrane"
seed 0

[ space ]
dimensions 40, 40, 40
resolution 0.5

[ includes ]
"topology/martini_v3.0.0.itp"
"topology/martini_v3.0.0_ions_v1.itp"
"topology/martini_v3.0.0_solvents_v1.itp"
"topology/martini_v3.0.0_phospholipids_v1.itp"
"topology/lysozyme.itp"
"topology/ubiquitin.itp"

[ compartments ]
membrane from "membrane_mask.npz"
solvent combines not membrane
close-to-membrane around 5 of membrane

[ segments ]
LYZ:lyz 300 from "structures/lysozyme.pdb" in solvent
UBQ:ubq 100 from "structures/ubiquitin.pdb" in close-to-membrane
'''
open('membrane_packing.bent', 'w').write(bent)
print(bent)


**Aşağıdaki hücre ne yapıyor?** Yerleşimi hesaplayıp koordinatları
üretmektedir. Bu aşamada yalnızca **proteinler** vardır; membran henüz dâhil
değildir.


In [ ]:
!bentopy-pack membrane_packing.bent placements.json
!bentopy-render placements.json packed_proteins.gro -t topol.top
print()
print(f'Paketlenmis proteinler: {parcacik_sayisi("packed_proteins.gro"):,} parcacik')


**Aşağıdaki hücre ne yapıyor?** Paketlenen proteinleri membran yapısıyla
birleştirip lipit sayısını topolojiye eklemektedir.

**Dikkat.** Birleştirilen membran `bentopy` tarafından üretilmediğinden
topolojide otomatik olarak yer almaz; lipit sayısı **elle** eklenmelidir. Bu,
iş akışının en kolay atlanan adımıdır ve atlanırsa `gmx grompp` koordinat ile
topoloji sayılarının uyuşmadığı hatasını verir.


In [ ]:
!bentopy-merge packed_proteins.gro structures/membrane.gro -o system.gro

# Membrandaki lipit sayisi topolojiye elle eklenir
res_m, ad_m, _ = koordinat_oku('structures/membrane.gro')
n_popc = int((np.isin(res_m, list(LIPIT)) & (ad_m == 'PO4')).sum())
print(f'Membrandaki POPC sayisi (PO4 merkezlerinden): {n_popc}')
print('Ogretim materyalindeki deger: 5408')

with open('topol.top', 'a') as f:
    f.write(f'POPC    {n_popc}\n')

print()
print('--- topol.top son satirlar ---')
print('\n'.join(open('topol.top').read().splitlines()[-6:]))


**Aşağıdaki hücre ne yapıyor?** Sistemi solvatlamaktadır. Bu, Drive'a
kaydedilecek olan ana sistemdir.


In [ ]:
!bentopy-solvate -i system.gro -o solvated_system.gro -t topol.top \
    -s NA:0.15M -s CL:0.15M --charge neutral

print()
print(f'Solvatlanmis sistem: {parcacik_sayisi("solvated_system.gro"):,} parcacik')
boyut_yaz('solvated_system.gro')


**Aşağıdaki hücre ne yapıyor?** Sistemden etkileşimli gösterime uygun
seyreltilmiş bir örnek üretip göstermektedir. Görünümü fare ile döndürerek
kalabalığı inceleyiniz.

Her protein türü **ayrı renkte** gösterilmektedir; renk ataması hücrenin
çıktısında listelenmektedir. Öğretim materyaliyle aynı düzen kullanılmaktadır:
yeşil lizozim, mavi ubikitin, gri membran.

> Bu, sistemin **örneklenmiş** bir gösterimidir; proteinlerden yalnızca omurga,
> lipitlerden yalnızca fosfat merkezleri alınmakta ve bunların bir kısmı
> çizilmektedir. Gerçek sistem çok daha yoğundur; tam sistem iki ve üç boyutlu
> çizimlerle incelenmektedir.


In [ ]:
seyreltilmis_gro('system.gro', 'gorunum_u2.gro',
                 protein_orani=0.12, lipit_orani=0.05)
yapi_goster('gorunum_u2.gro', stil='sistem')


**Aşağıdaki hücre ne yapıyor?** Her molekülün ağırlık merkezini hesaplayıp üç
boyutlu perspektifle çizmektedir. Bu görünüm, moleküllerin kutu içindeki
dağılımını tek bakışta göstermektedir.

Renk düzeni öğretim materyaliyle aynıdır: yeşil lizozim, mavi ubikitin, gri
membran. Simülasyon kutusu tel çerçeve olarak gösterilmektedir.


In [ ]:
uc_boyut_goster('system.gro',
                os.path.join(D_GORSEL, 'uygulama2_3boyut.png'),
                'Uygulama 2 - molekul merkezleri (uc boyutlu)')


**Aşağıdaki hücre ne yapıyor?** Sistemin kesitini çizmektedir.

**Beklenen görünüm:** lipitler (turuncu) kutunun ortasında dar bir bant
oluşturmalı, protein (mavi) dağılımı membran çevresinde belirgin biçimde
**zenginleşmelidir**. Bu, `close-to-membrane` bölme tanımının işlediğinin
doğrudan görsel kanıtıdır.


In [ ]:
kesit_ciz('solvated_system.gro',
          os.path.join(D_GORSEL, 'uygulama2_kesit.png'),
          'Uygulama 2 - membran cevresinde konuma bagli paketleme',
          dilim=6.0, eksen=('x', 'z'))


**Aşağıdaki hücre ne yapıyor?** Konumsal kuralın işlediğini **sayısal olarak**
sınamaktadır.

Ölçüt, iki protein türü için **ayrı ayrı** hesaplanmaktadır: membran yüzeyinden
5 nm'lik kabuktaki yoğunluk, kabuğun dışındaki yoğunluğa bölünmektedir. Beklenen
sonuç:

- **Ubikitin** yalnızca `close-to-membrane` bölmesine yerleştirildiğinden kabukta
  belirgin biçimde zenginleşmelidir
- **Lizozim** tüm çözücü hacmine dağıtıldığından oranı 1'e yakın olmalıdır

İki protein birlikte sayılırsa 300 lizozim, 100 ubikitinin sinyalini bastırır ve
ölçüt anlamsızlaşır; bu nedenle ayrı hesaplanmaktadır.


In [ ]:
res, ad, xyz = koordinat_oku('solvated_system.gro')

z_lipit = xyz[np.isin(res, list(LIPIT)), 2]
mz   = z_lipit.mean()
yari = (z_lipit.max() - z_lipit.min()) / 2
kabuk = 5.0                       # .bent dosyasindaki "around 5 of membrane"
z_min, z_max = xyz[:, 2].min(), xyz[:, 2].max()

hacim_yakin = 2 * kabuk
hacim_uzak  = (z_max - z_min) - 2 * (yari + kabuk)

print(f'Membran merkezi        : z = {mz:.1f} nm')
print(f'Membran yari kalinligi : {yari:.1f} nm')
print(f'Incelenen kabuk        : membran yuzeyinden {kabuk:.0f} nm')
print()

sonuc = {}
for etiket in ('UBQ', 'LYZ'):
    m = (res == etiket)
    if m.sum() == 0:
        continue
    z = xyz[m, 2]
    d = np.abs(z - mz)                       # membran merkezine uzaklik
    yakin = (d > yari) & (d <= yari + kabuk)  # membran disi, kabuk icinde
    uzak  = d > yari + kabuk
    yog_yakin = yakin.sum() / hacim_yakin
    yog_uzak  = uzak.sum()  / max(hacim_uzak, 1e-9)
    oran = yog_yakin / max(yog_uzak, 1e-9)
    sonuc[etiket] = oran
    print(f'{etiket}: kabukta {yakin.sum():>7,} | uzakta {uzak.sum():>7,} '
          f'| zenginlesme orani {oran:.2f}')

print()
ubq, lyz = sonuc.get('UBQ'), sonuc.get('LYZ')
if ubq is not None and lyz is not None:
    print(f'UBQ / LYZ zenginlesme farki: {ubq/max(lyz, 1e-9):.1f} kat')
    print()
    if ubq > 2 * lyz:
        print('DOGRULAMA BASARILI: ubikitin membran yuzeyinde belirgin bicimde')
        print('zenginlesmis, lizozim ise homojen dagilmistir. Bu, .bent dosyasindaki')
        print('close-to-membrane bolme tanimin isledigini gostermektedir.')
    else:
        print('UYARI: beklenen zenginlesme gorulmedi.')
        print('.bent dosyasindaki [ compartments ] bolumu kontrol edilmelidir.')
else:
    print('UYARI: UBQ veya LYZ bulunamadi; rezidu adlari kontrol edilmelidir.')


**Aşağıdaki hücre ne yapıyor?** z profilini çizmektedir. Protein eğrisinin
lipit bandının iki yanında tepe yapması beklenir.


In [ ]:
z_profili('solvated_system.gro',
          os.path.join(D_GORSEL, 'uygulama2_z_profili.png'),
          'Uygulama 2 - z ekseni boyunca dagilim')


**Aşağıdaki hücre ne yapıyor?** Kurduğumuz sistemi öğretim
materyalindeki referans görüntüyle yan yana koymaktadır. Soldaki çizim
bu not defterinde üretilmiştir; sağdaki, cgmartini.nl sayfasındaki VMD
render'ıdır.

Referans görüntüde membran (gri) kutuyu ikiye ayırmakta, lizozim (yeşil)
çözücü hacmine dağılmakta, ubikitin (mavi) ise **membran yüzeyine yakın**
bir tabaka oluşturmaktadır. Ubikitinin bu birikimi `close-to-membrane`
bölme tanımının sonucudur ve kendi kesitinizde de görülmelidir.


In [ ]:
referans_karsilastir('solvated_system.gro',
                     'figure_tutorial_2.png',
                     os.path.join(D_GORSEL, 'uygulama2_karsilastirma.png'),
                     'Uygulama 2 - membran cevresinde konuma bagli paketleme')


**Aşağıdaki hücre ne yapıyor?** Uygulama 2 dosyalarını Drive'a
kaydetmektedir. Bu uygulamanın solvatlanmış sistemi, boyutu ne olursa olsun
kaydedilmektedir; 10. bölümdeki simülasyon paketinin temeli budur.


In [ ]:
kaydet(['membrane_packing.bent', 'placements.json', 'membrane_mask.npz',
        'labels.gro', 'packed_proteins.gro', 'system.gro', 'topol.top'],
       D_UYG2)
kaydet(['solvated_system.gro'], D_UYG2, buyuk_dahil=True)


---
## 8. Uygulama 3 — Çok bölmeli sistem

Çift membranla ayrılmış **iki bölme** tanımlanmakta ve her bölmeye farklı
protein yerleştirilmektedir. Bu, bir organeli veya iki hücre bölmesini temsil
eden en gerçekçi kurulumdur.

Süre elverdiği takdirde yürütülecektir.

**Aşağıdaki hücre ne yapıyor?** Çift membran yapısından üç ayrı maske
üretmektedir:

| Bayrak | Ürettiği |
|---|---|
| `-b compartment_labels.gro` | Bölme etiketlerinin görsel denetimi için yapı dosyası |
| `-l -1:A_mask.npz` | A bölmesi (membranın bir yanı) |
| `-l -2:B_mask.npz` | B bölmesi (diğer yanı) |
| `-l 1,2:membrane_mask.npz` | İki membranın kendisi |


In [ ]:
!bentopy-mask structures/double_membrane.gro -b compartment_labels.gro
!bentopy-mask structures/double_membrane.gro \
    -l  -1:A_mask.npz \
    -l  -2:B_mask.npz \
    -l 1,2:membrane_mask.npz
print()
boyut_yaz('A_mask.npz', 'B_mask.npz', 'membrane_mask.npz',
          'compartment_labels.gro')
print()
eksik_maske = [m for m in ('A_mask.npz', 'B_mask.npz', 'membrane_mask.npz')
               if not os.path.exists(m)]
print('DOGRULAMA BASARILI: uc maske de uretildi.' if not eksik_maske
      else f'UYARI: eksik maskeler -> {eksik_maske}')


**Aşağıdaki hücre ne yapıyor?** Çift membran yapısının kesitini çizmektedir.
İki lipit bandı ve aralarındaki bölmeler görülmelidir.


In [ ]:
kesit_ciz('structures/double_membrane.gro',
          os.path.join(D_GORSEL, 'uygulama3_cift_membran.png'),
          'Girdi yapisi - cift membran', dilim=6.0, eksen=('x', 'z'))


**Aşağıdaki hücre ne yapıyor?** Uygulama 3'ün yapılandırma dosyasını
yazmaktadır. Bölme tanımlarında iki yeni ifade kullanılmaktadır:

```
membrane-neighborhood around 4 of membrane
B-close-to-membrane combines membrane-neighborhood and B
```

İkincisi bir **kesişim** tanımlar: hem B bölmesinde hem membrana yakın olan
hacim. Böylece ubikitin yalnızca B tarafındaki membran yüzeyine yerleşmekte,
A tarafına geçmemektedir.


In [ ]:
bent = '''[ general ]
title "Proteins in different compartments"
seed 0

[ space ]
dimensions 40, 40, 40
resolution 0.5

[ includes ]
"topology/martini_v3.0.0.itp"
"topology/martini_v3.0.0_ions_v1.itp"
"topology/martini_v3.0.0_solvents_v1.itp"
"topology/martini_v3.0.0_phospholipids_v1.itp"
"topology/lysozyme.itp"
"topology/ubiquitin.itp"

[ compartments ]
membrane from "membrane_mask.npz"
A from "A_mask.npz"
B from "B_mask.npz"
membrane-neighborhood around 4 of membrane
B-close-to-membrane combines membrane-neighborhood and B

[ segments ]
LYZ:lyz 200 from "structures/lysozyme.pdb" in A
UBQ:ubq 100 from "structures/ubiquitin.pdb" in B-close-to-membrane
'''
open('compartment_packing.bent', 'w').write(bent)
print(bent)


**Aşağıdaki hücre ne yapıyor?** Yerleşimi hesaplayıp koordinatları üretmekte,
çift membranla birleştirmekte ve lipit sayısını topolojiye eklemektedir.


In [ ]:
!bentopy-pack compartment_packing.bent placements.json
!bentopy-render placements.json packed_proteins.gro -t topol.top
!bentopy-merge packed_proteins.gro structures/double_membrane.gro -o system.gro

res_m, ad_m, _ = koordinat_oku('structures/double_membrane.gro')
n_popc = int((np.isin(res_m, list(LIPIT)) & (ad_m == 'PO4')).sum())
print()
print(f'Cift membrandaki POPC sayisi: {n_popc}  (ogretim materyali: 10816)')
with open('topol.top', 'a') as f:
    f.write(f'POPC    {n_popc}\n')


**Aşağıdaki hücre ne yapıyor?** Sistemi solvatlamaktadır.


In [ ]:
!bentopy-solvate -i system.gro -o solvated_system.gro -t topol.top \
    -s NA:0.15M -s CL:0.15M --charge neutral

print()
print(f'Cok bolmeli sistem: {parcacik_sayisi("solvated_system.gro"):,} parcacik')
boyut_yaz('solvated_system.gro')


**Aşağıdaki hücre ne yapıyor?** Sistemden etkileşimli gösterime uygun
seyreltilmiş bir örnek üretip göstermektedir. Görünümü fare ile döndürerek
kalabalığı inceleyiniz.

Her protein türü **ayrı renkte** gösterilmektedir; renk ataması hücrenin
çıktısında listelenmektedir. Öğretim materyaliyle aynı düzen kullanılmaktadır:
yeşil lizozim, mavi ubikitin, gri membran.

> Bu, sistemin **örneklenmiş** bir gösterimidir; proteinlerden yalnızca omurga,
> lipitlerden yalnızca fosfat merkezleri alınmakta ve bunların bir kısmı
> çizilmektedir. Gerçek sistem çok daha yoğundur; tam sistem iki ve üç boyutlu
> çizimlerle incelenmektedir.


In [ ]:
seyreltilmis_gro('system.gro', 'gorunum_u3.gro',
                 protein_orani=0.12, lipit_orani=0.05)
yapi_goster('gorunum_u3.gro', stil='sistem')


**Aşağıdaki hücre ne yapıyor?** Her molekülün ağırlık merkezini hesaplayıp üç
boyutlu perspektifle çizmektedir. Bu görünüm, moleküllerin kutu içindeki
dağılımını tek bakışta göstermektedir.

Renk düzeni öğretim materyaliyle aynıdır: yeşil lizozim, mavi ubikitin, gri
membran. Simülasyon kutusu tel çerçeve olarak gösterilmektedir.


In [ ]:
uc_boyut_goster('system.gro',
                os.path.join(D_GORSEL, 'uygulama3_3boyut.png'),
                'Uygulama 3 - molekul merkezleri (uc boyutlu)')


**Aşağıdaki hücre ne yapıyor?** Sistemin kesitini çizmektedir.

**Beklenen görünüm:** **iki** lipit bandı ve bunlar arasında bölmeye özgü
protein dağılımı. Lizozim yalnızca A bölmesinde, ubikitin ise B bölmesinin
membrana yakın kesiminde bulunmalıdır.


In [ ]:
kesit_ciz('solvated_system.gro',
          os.path.join(D_GORSEL, 'uygulama3_kesit.png'),
          'Uygulama 3 - cift membranli cok bolmeli sistem',
          dilim=6.0, eksen=('x', 'z'))


**Aşağıdaki hücre ne yapıyor?** z profilini çizmektedir. İki lipit tepesi ve
protein dağılımının bölmelere ayrışması bu grafikte görülmelidir.


In [ ]:
z_profili('solvated_system.gro',
          os.path.join(D_GORSEL, 'uygulama3_z_profili.png'),
          'Uygulama 3 - z ekseni boyunca dagilim')


**Aşağıdaki hücre ne yapıyor?** Kurduğumuz sistemi öğretim
materyalindeki referans görüntüyle yan yana koymaktadır. Soldaki çizim
bu not defterinde üretilmiştir; sağdaki, cgmartini.nl sayfasındaki VMD
render'ıdır.

Referans görüntüde **iki** membran iki ayrı bölme oluşturmaktadır. Ubikitin
(mavi) yalnızca membranlar arasındaki bölmede, lizozim (yeşil) ise diğer
bölmede bulunmaktadır. Bölmelerin birbirine karışmamış olması, maske ve
kesişim tanımlarının doğru çalıştığını göstermektedir.


In [ ]:
referans_karsilastir('solvated_system.gro',
                     'figure_tutorial_3.png',
                     os.path.join(D_GORSEL, 'uygulama3_karsilastirma.png'),
                     'Uygulama 3 - cift membranli cok bolmeli sistem')


**Aşağıdaki hücre ne yapıyor?** Uygulama 3 dosyalarını Drive'a
kaydetmektedir.


In [ ]:
kaydet(['compartment_packing.bent', 'placements.json',
        'compartment_labels.gro', 'A_mask.npz', 'B_mask.npz',
        'membrane_mask.npz', 'packed_proteins.gro', 'system.gro',
        'topol.top', 'solvated_system.gro'], D_UYG3)


---
## 9. Ölçek karşılaştırması

**Aşağıdaki hücre ne yapıyor?** Kurs boyunca kurulan sistemlerin
büyüklüklerini karşılaştırmaktadır. Oturum 4'ün çıktısı Drive'da bulunuyorsa
o da tabloya dâhil edilmektedir.


In [ ]:
kayitlar = []

o4 = os.path.join(DRIVE_KOK, 'martini_input', 'simulasyon', 'sistem.gro')
if os.path.exists(o4):
    kayitlar.append(('Oturum 4 - tek reseptor / POPC', parcacik_sayisi(o4)))

for etiket, klasor in [('Uygulama 1 - kutuda 650 lizozim', D_UYG1),
                       ('Uygulama 2 - membran + proteinler', D_UYG2),
                       ('Uygulama 3 - cok bolmeli sistem', D_UYG3)]:
    yol = os.path.join(klasor, 'solvated_system.gro')
    if os.path.exists(yol):
        kayitlar.append((etiket, parcacik_sayisi(yol)))

if kayitlar:
    print(f"{'Sistem':<38}{'Parcacik':>14}")
    print('-' * 52)
    for e, n in kayitlar:
        print(f'{e:<38}{n:>14,}')
    print()
    en_kucuk = min(n for _, n in kayitlar)
    en_buyuk = max(n for _, n in kayitlar)
    print(f'Olcek farki: {en_buyuk/en_kucuk:.0f} kat')
else:
    print('Karsilastirilacak sistem bulunamadi.')

print()
print('Not: Uygulama 1 ve 3 icin buyuk dosyalar varsayilan olarak')
print('kaydedilmediginden tabloda gorunmeyebilir (bkz. BUYUK_DOSYA_KAYDET).')


---
## 10. Simülasyon paketinin hazırlanması

Bu bölümde, GROMACS ile **doğrudan çalıştırılabilecek** eksiksiz bir dosya
kümesi oluşturulmaktadır. Oturum 4'teki paketle aynı düzendedir.

Paket **Uygulama 2** (membran + proteinler) sistemi üzerine kurulmaktadır; bu,
üç uygulama arasında hem membran hem çözünür protein içeren en temsili
sistemdir.

| Dosya | İşlevi |
|---|---|
| `solvated_system.gro` | Başlangıç koordinatları |
| `topol.top` | Topoloji |
| `topology/*.itp` | Martini 3 kuvvet alanı ve protein parametreleri |
| `em.mdp`, `eq.mdp`, `md.mdp` | Simülasyon parametreleri |
| `index.ndx` | Termostat grupları |
| `calistir.sh` | Üç aşamayı sırayla yürüten betik |
| `OKUBENI.md` | Kullanım açıklaması |

**Aşağıdaki hücre ne yapıyor?** Uygulama 2'nin çıktılarını geçici bir paket
klasörüne toplamaktadır. Öğretim arşiviyle gelen `.mdp` dosyaları ve topoloji
dosyaları da kopyalanmaktadır.


In [ ]:
PAKET = '/content/paket'
if os.path.exists(PAKET):
    shutil.rmtree(PAKET)
os.makedirs(PAKET)

# Uygulama 2 ciktilari (Drive'dan geri alinir)
for f in ('solvated_system.gro', 'topol.top'):
    kaynak = os.path.join(D_UYG2, f)
    if os.path.exists(kaynak):
        shutil.copy(kaynak, PAKET)
        print(f'  {f:<24} alindi')
    else:
        print(f'  {f:<24} BULUNAMADI - 7. bolum calistirilmis mi?')

# Kuvvet alani ve protein topolojileri
os.makedirs(os.path.join(PAKET, 'topology'), exist_ok=True)
for f in glob.glob(os.path.join(CALISMA, 'topology', '*.itp')):
    shutil.copy(f, os.path.join(PAKET, 'topology'))

# Ogretim arsiviyle gelen mdp dosyalari
os.makedirs(os.path.join(PAKET, 'mdp_files'), exist_ok=True)
for f in glob.glob(os.path.join(CALISMA, 'mdp_files', '*.mdp')):
    shutil.copy(f, os.path.join(PAKET, 'mdp_files'))

print()
print('Paket icerigi:')
for kok, _, dosyalar in os.walk(PAKET):
    for d in sorted(dosyalar):
        y = os.path.join(kok, d)
        print(f'  {os.path.relpath(y, PAKET):<44} {boyut_str(os.path.getsize(y)):>10}')


**Aşağıdaki hücre ne yapıyor?** Termostat gruplarını tanımlayan `index.ndx`
dosyasını doğrudan koordinat dosyasından üretmektedir.

`eq.mdp` ve `md.mdp` dosyaları `tc-grps = Protein Lipid Solvent` satırını
içermektedir; bu adlarda üç grubun tanımlı olması gerekir. GROMACS'in öntanımlı
grupları arasında `Lipid` ve `Solvent` bulunmadığından bunlar oluşturulmalıdır.

**Neden `gmx make_ndx` kullanılmıyor?** `make_ndx` etkileşimli bir araçtır ve
komutları grup **numaralarına** göre yorumlar. `name 0 Protein` gibi bir komut,
yeni oluşturulan grubu değil, 0 numaralı grubu (yani `System`'i) yeniden
adlandırır. Grup numaraları sisteme göre değiştiğinden bu yaklaşım sessizce
yanlış gruplar üretebilmektedir. Aşağıdaki fonksiyon grupları rezidü adlarından
doğrudan belirlediği için bu belirsizliği ortadan kaldırmakta ve sonucu
doğrulamaktadır.

**Neden ayrı gruplar?** Protein, lipit ve çözücünün ısı kapasiteleri farklı
olduğundan termostatın her birine ayrı uygulanması önerilmektedir; tek grup
kullanılması sıcaklık dengesizliğine yol açabilmektedir.


In [ ]:
def index_yaz(gro, ndx):
    """GRO dosyasindan dogrudan index.ndx yazar."""
    satirlar = open(gro).read().splitlines()
    n = int(satirlar[1])
    gruplar = {'Protein': [], 'Lipid': [], 'Solvent': []}
    for i, s in enumerate(satirlar[2:2+n], start=1):
        rn = s[5:10].strip()
        if rn in LIPIT:
            gruplar['Lipid'].append(i)
        elif rn in COZUCU or rn in IYON:
            gruplar['Solvent'].append(i)
        else:
            gruplar['Protein'].append(i)

    with open(ndx, 'w') as f:
        for ad_g, idx in gruplar.items():
            f.write(f'[ {ad_g} ]\n')
            for k in range(0, len(idx), 15):
                f.write(' '.join(f'{x:>7}' for x in idx[k:k+15]) + '\n')

    sayim = {k: len(v) for k, v in gruplar.items()}
    for g, s in sayim.items():
        print(f'  {g:<10} {s:>10,} parcacik')
    toplam = sum(sayim.values())
    print()
    print(f'  Gruplarin toplami : {toplam:,}')
    print(f'  Sistemdeki toplam : {n:,}')
    print(f'  index.ndx boyutu  : {boyut_str(os.path.getsize(ndx))}')
    print()
    if toplam == n and all(sayim.values()):
        print('DOGRULAMA BASARILI: uc grup sistemi tam ve ortusmeden kapsiyor.')
    elif not all(sayim.values()):
        bos = [g for g, s in sayim.items() if s == 0]
        print(f'UYARI: bos grup(lar) var -> {bos}')
    else:
        print('UYARI: gruplarin toplami sistemle uyusmuyor.')
    return sayim

print('index_yaz() hazir.')


**Aşağıdaki hücre ne yapıyor?** `index.ndx` dosyasını üretip grupların
sistemi eksiksiz kapsadığını doğrulamaktadır.


In [ ]:
index_yaz(os.path.join(PAKET, 'solvated_system.gro'), os.path.join(PAKET, 'index.ndx'))


**Aşağıdaki hücre ne yapıyor?** Çalıştırma betiğini ve açıklama dosyasını
yazmaktadır. Betik, öğretim materyalindeki üç aşamayı sırayla yürütmektedir.


In [ ]:
calistir = '''#!/usr/bin/env bash
#
# Biyofizik 2026 Kursu - Oturum 6
# Kalabalik membran sistemi (bentopy Uygulama 2) - Martini 3
#
# Kullanim:  bash calistir.sh
#
set -euo pipefail

# --- 1. Enerji minimizasyonu ---
gmx grompp -f mdp_files/em.mdp -c solvated_system.gro -p topol.top \\
    -o em.tpr -maxwarn 5
gmx mdrun -v -deffnm em

# --- 2. Dengeleme ---
gmx grompp -f mdp_files/eq.mdp -c em.gro -p topol.top -n index.ndx \\
    -o eq.tpr -maxwarn 5
gmx mdrun -v -deffnm eq

# --- 3. Uretim simulasyonu ---
# DIKKAT: bu boyutta bir sistem GPU uzerinde bile gunler surebilir.
gmx grompp -f mdp_files/md.mdp -c eq.gro -p topol.top -n index.ndx \\
    -o md.tpr -maxwarn 5
gmx mdrun -v -deffnm md
'''

gro = os.path.join(PAKET, 'solvated_system.gro')
n_toplam = parcacik_sayisi(gro) if os.path.exists(gro) else 0

okubeni = f'''# Kalabalik Membran Sistemi - Martini 3

Biyofizik 2026 Kursu, Oturum 6 ciktisi (bentopy Uygulama 2).

## Sistem

- Membran : POPC cift tabaka (hazir yapi)
- Protein : 300 lizozim (cozucu hacminde),
            100 ubikitin (membrana 5 nm yakinlikta)
- Kutu    : 40 x 40 x 40 nm
- Cozucu  : Martini standart su (W), 0.15 M NaCl
- Toplam  : {n_toplam:,} parcacik

## Calistirma

```bash
bash calistir.sh
```

Betik uc asamayi sirayla yurutur: enerji minimizasyonu, dengeleme, uretim.

## Gereksinimler

- GROMACS 2021 veya uzeri
- Bu boyutta bir sistem icin GPU ve tercihen coklu dugum gereklidir

## Notlar

- Termostat gruplari `index.ndx` icinde tanimlidir: Protein, Lipid, Solvent
- Membran sistemi oldugu icin barostat `semiisotropic` ayarlanmistir
- Sistem `bentopy` ile kuruldugundan iyon adlari Martini 3 ile uyumludur
  (NA, CL) - `insane` ciktilarindaki gibi duzeltme gerekmez
- Analiz oncesinde periyodik sinir kosullari duzeltilmelidir:
  `gmx trjconv -s md.tpr -f md.xtc -o md_nojump.xtc -pbc nojump`

## Kaynak

Ogretim materyali: https://cgmartini.nl/docs/tutorials/Martini3/Bentopy/
Kurs deposu     : https://github.com/eygpcr/biyofizik2026-martini
'''

open(os.path.join(PAKET, 'calistir.sh'), 'w').write(calistir)
open(os.path.join(PAKET, 'OKUBENI.md'), 'w').write(okubeni)
print('calistir.sh ve OKUBENI.md yazildi.')


**Aşağıdaki hücre ne yapıyor?** Paketin gerçekten çalıştırılabilir olduğunu
sınamaktadır: paket klasörüne geçip yalnızca oradaki dosyalarla `gmx grompp`
çalıştırmaktadır.

**İki aşama da sınanmaktadır:**

1. `em.mdp` — enerji minimizasyonu; `index.ndx` gerektirmez
2. `eq.mdp` — dengeleme; `tc-grps = Protein Lipid Solvent` satırı nedeniyle
   `index.ndx` dosyasını ve grupların doğruluğunu sınar

Yalnızca birincisinin sınanması yeterli değildir; termostat gruplarındaki bir
hata ancak ikinci aşamada ortaya çıkmaktadır.

> Bu boyutta bir sistem için `grompp` birkaç dakika sürebilir ve önemli miktarda
> bellek kullanır.


In [ ]:
%cd {PAKET}
print('--- 1) em.mdp (index.ndx gerektirmez) ---')
!gmx grompp -f mdp_files/em.mdp -c solvated_system.gro -p topol.top \
    -o dogrulama_em.tpr -maxwarn 10 2>&1 | tail -6

print()
print('--- 2) eq.mdp (termostat gruplarini kullanir) ---')
!gmx grompp -f mdp_files/eq.mdp -c solvated_system.gro -p topol.top \
    -n index.ndx -o dogrulama_eq.tpr -maxwarn 10 2>&1 | tail -8
%cd {CALISMA}

em = os.path.join(PAKET, 'dogrulama_em.tpr')
eq = os.path.join(PAKET, 'dogrulama_eq.tpr')
print()
if os.path.exists(em) and os.path.exists(eq):
    print('DOGRULAMA BASARILI: paket kendi basina calistirilabilir.')
    print('Hem enerji minimizasyonu hem dengeleme girdisi uretilebildi;')
    print('termostat gruplari dogru tanimlanmis.')
elif os.path.exists(em):
    print('KISMI BASARI: em.tpr uretildi ancak eq.tpr uretilemedi.')
    print('index.ndx veya tc-grps ayarlari kontrol edilmelidir.')
else:
    print('UYARI: paket eksik olabilir; yukaridaki hata incelenmelidir.')

for f in (em, eq):
    if os.path.exists(f):
        os.remove(f)


---
## 11. Drive klasörünün özeti

**Aşağıdaki hücre ne yapıyor?** Bu oturumda üretilen tüm dosyaları
listelemektedir.


In [ ]:
print('Google Drive icerigi:',
      OTURUM.replace('/content/drive/MyDrive', "Drive'im"))
print()
genel = 0
for alt in ('uygulama1_kutu', 'uygulama2_membran', 'uygulama3_bolmeler',
            'gorseller', 'simulasyon'):
    yol = os.path.join(OTURUM, alt)
    if not os.path.isdir(yol):
        continue
    dosyalar = []
    for kok, _, ds in os.walk(yol):
        for d in ds:
            dosyalar.append(os.path.join(kok, d))
    boyut = sum(os.path.getsize(d) for d in dosyalar)
    genel += boyut
    print(f'{alt}/ ({len(dosyalar)} dosya, {boyut_str(boyut)})')
    for d in sorted(dosyalar):
        print(f'    {os.path.relpath(d, yol)}')
    print()
print(f'Toplam: {boyut_str(genel)}')


### İsteğe bağlı: paketi bilgisayara indirme

**Aşağıdaki hücre ne yapıyor?** Simülasyon paketini tek arşiv hâlinde
indirmektedir. Dosyalar zaten Drive'da olduğundan bu adım gerekli değildir.


In [ ]:
# import shutil
# from google.colab import files
# arsiv = shutil.make_archive('/content/bentopy_paket', 'zip', D_SIM)
# files.download(arsiv)


---
## Sık karşılaşılan sorunlar

| Sorun | Nedeni | Çözümü |
|---|---|---|
| `bentopy-pack: command not found` | Kurulum tamamlanmamış | 2. bölümdeki kurulum hücresi tekrarlanmalıdır |
| `pip install bentopy` derleme hatası veriyor | Platform için hazır paket yok (örneğin macOS) | Colab kullanılmalı veya [rustup](https://rustup.rs/) ile Rust kurulmalıdır |
| Maske dosyası üretilmiyor | Çalışma dizini yanlış | `os.chdir('/content/tutorial_files')` çalıştırılmalıdır |
| Paketleme çok uzun sürüyor | Kopya sayısı yüksek | `[ segments ]` bölümündeki sayı azaltılabilir |
| Hedeflenen sayıda kopya yerleştirilemedi | Kutu dolmuş | Kutu büyütülmeli veya kopya sayısı azaltılmalıdır |
| `number of coordinates does not match topology` | `bentopy-merge` sonrası lipit sayısı eklenmemiş | 7. bölümdeki topolojiye ekleme adımı çalıştırılmalıdır |
| `Group Lipid not found` | `index.ndx` eksik veya hatalı | 10. bölüm yeniden çalıştırılmalıdır |
| Oturum belleği doluyor | Solvatlanmış sistem çok büyük | Çalışma zamanı yeniden başlatılıp kutu küçültülmelidir |
| Drive'da yer kalmadı | Büyük koordinat dosyaları | `BUYUK_DOSYA_KAYDET = False` bırakılmalı, eski klasörler silinmelidir |
| py3Dmol görünümü donuyor | Seyreltme oranı yüksek | `protein_orani` ve `lipit_orani` düşürülmelidir |

---

## Kaynaklar

- [Bentopy Tutorial — cgmartini.nl](https://cgmartini.nl/docs/tutorials/Martini3/Bentopy/)
- [bentopy deposu](https://github.com/marrink-lab/bentopy) ve [wiki](https://github.com/marrink-lab/bentopy/wiki)
- [`.bent` dosya biçimi başvurusu](https://github.com/marrink-lab/bentopy/wiki/Reference-for-bent)
- [`bentopy-solvate` belgelendirmesi](https://github.com/marrink-lab/bentopy/blob/main/src/solvate/README.md)
- [Protein kompleksleri — Martini](https://cgmartini.nl/docs/tutorials/Martini3/ProteinsIIb/)
- [TS2CG v2.0](https://github.com/weria-pezeshkian/TS2CG-v2.0/wiki/Tutorial) — vezikül ve karmaşık geometriler

Ayrıca bkz. [`ILERI_OKUMA.md`](https://github.com/eygpcr/biyofizik2026-martini/blob/main/ILERI_OKUMA.md)
